# Efficient Analysis — Projection Geometry First

Ordered to minimise generation cost:

| Phase | What it does | Generation? |
|-------|--------------|-------------|
| **1. Existing data** | SVD + validation metrics already in manifest | None |
| **2. Gate: step-count validation** | Does variance trajectory shape survive at 4–8 steps? | ~100 images |
| **3+. Deferred stubs** | Described in markdown; coded once gate passes | TBD |

Texture-coloring (Exp 8 corpus swap) is the lowest-priority item and is **not coded here**.
It will be added after all other experiments have produced data to compare against,
to ensure the corpus retrain does not invalidate prior findings.

---
**Codebase prerequisite (already applied):**  
`experiment.py:461` now reads `num_inference_steps` from the config JSON (default 30).
Pass `"num_inference_steps": 8` in a config to get a fast survey pass.

## 1 · Setup

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))

REPO_URL    = 'https://github.com/leonorae/slicer'
REPO_BRANCH = 'claude/analyze-experiment-confounders-uBu9h'

if IN_COLAB:
    REPO_DIR    = Path('/content/slicer')
    DRIVE_BASE  = Path('/content/drive/MyDrive/diffusion_microscope')
    RESULTS_DIR = DRIVE_BASE / 'experiment_results_efficient'
    HF_CACHE    = DRIVE_BASE / 'hf_cache'
else:
    REPO_DIR    = Path('/home/user/slicer')
    DRIVE_BASE  = None
    RESULTS_DIR = REPO_DIR / 'experiment_results'
    HF_CACHE    = None

# Existing results directory — where the trained manifest already lives.
# Override this if your results are elsewhere.
EXISTING_RESULTS = RESULTS_DIR

print(f'IN_COLAB       : {IN_COLAB}')
print(f'REPO_DIR       : {REPO_DIR}')
print(f'EXISTING_RESULTS: {EXISTING_RESULTS}')

In [ ]:
# Colab only — mount Drive, clone repo, install deps
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU')

    if REPO_DIR.is_dir():
        !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
        !git -C {REPO_DIR} checkout {REPO_BRANCH}
        !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
    else:
        !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

    !pip install -q open-clip-torch diffusers Pillow lpips datasets nltk sentencepiece accelerate scikit-learn
    !pip install -q -e {REPO_DIR} --no-deps

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    HF_CACHE.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(HF_CACHE)
    os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

---
## Phase 1 — Existing projection geometry (no generation)

The manifest contains per-layer SVD and validation metrics for whatever alpha values
were trained, computed during the training phase.
No images, no LLM re-runs, no SD — pure manifold geometry.

**What we can learn here:**
- How erank and condition number vary with layer depth and regularisation strength
- Where the R² ↔ nn_recall@5 trade-off is sharpest (those layers are most sensitive to alpha choice)
- Which alpha values are worth running in the gating experiment (skip those that collapse or are numerically identical)

In [ ]:
manifest_path = EXISTING_RESULTS / 'manifest.json'
if not manifest_path.exists():
    print(f'No manifest found at {manifest_path}')
    print('If this is a fresh run, point EXISTING_RESULTS at a directory with a completed train phase.')
    manifest = {}
else:
    with open(manifest_path) as f:
        manifest = json.load(f)

projections = manifest.get('projections', {})
print('Projection keys:', list(projections.keys()))
print('Manifest top-level keys:', [k for k in manifest if k != 'projections'])

In [ ]:
# Discover projection keys from the manifest — works for any alpha set.
import re as _re

def _parse_alpha(pk):
    """Extract alpha label from a proj_key like per_layer_alpha1000 or per_layer_alphaauto."""
    m = _re.search(r'alpha(.+)$', pk)
    if not m:
        return pk
    raw = m.group(1)
    try:
        v = float(raw)
        return int(v) if v == int(v) else v
    except ValueError:
        return raw   # 'auto' etc.

# Build PROJ_KEYS from whatever is in this manifest
PROJ_KEYS = sorted(projections.keys())

if not PROJ_KEYS:
    print('No projection keys found in manifest — was the train phase run?')
else:
    print(f'Found {len(PROJ_KEYS)} projection(s): {PROJ_KEYS}')

# Assign colours by position so any number of alphas renders cleanly
_PALETTE = ['#E53935', '#FB8C00', '#FDD835', '#43A047', '#1E88E5',
            '#8E24AA', '#00ACC1', '#6D4C41']
ALPHA_COLORS = {pk: _PALETTE[i % len(_PALETTE)] for i, pk in enumerate(PROJ_KEYS)}
ALPHA_LABELS = {pk: f'α={_parse_alpha(pk)}' for pk in PROJ_KEYS}


def get_metric(proj_key, metric_group, metric_name):
    """Return list of (layer_idx, value) sorted by layer, skipping +inf (singular)."""
    proj  = projections.get(proj_key, {})
    group = proj.get(metric_group, {})
    pairs = []
    for layer_str, vals in group.items():
        try:
            layer = int(layer_str)
            v = vals.get(metric_name)
            if v is not None and not (isinstance(v, float) and (np.isinf(v) or np.isnan(v))):
                pairs.append((layer, v))
        except (ValueError, AttributeError, TypeError):
            pass
    return sorted(pairs)


# Sanity table
print(f'\n{"proj_key":35s}  {"L0 erank":>8}  {"Lmax erank":>10}  {"L0 cond":>10}  {"L0 R²":>6}  {"Lmax nn@5":>9}')
for pk in PROJ_KEYS:
    erank = dict(get_metric(pk, 'svd', 'erank'))
    cond  = dict(get_metric(pk, 'svd', 'condition_number'))
    r2    = dict(get_metric(pk, 'validation', 'projection_r2'))
    nnr   = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
    if not erank:
        print(f'{pk:35s}  (no SVD data)')
        continue
    L0   = min(erank)
    Lmax = max(erank)
    cond_v = cond.get(L0)
    cond_s = f'{cond_v:.2e}' if cond_v is not None else '       ?'
    print(f'{pk:35s}  {erank.get(L0, 0):8.0f}  {erank.get(Lmax, 0):10.0f}  '
          f'{cond_s:>10}  '
          f'{r2.get(L0, float("nan")):6.3f}  '
          f'{nnr.get(Lmax, float("nan")):9.3f}')

In [ ]:
# ── SVD trajectories: erank, condition_number, n_visible ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

SVD_METRICS = [
    ('erank',            'Effective rank (erank)',        False),
    ('condition_number', 'Condition number (log scale)',  True),
    ('n_visible',        'n_visible (non-zero SV dims)',  False),
]

for ax, (metric, ylabel, log_y) in zip(axes, SVD_METRICS):
    for pk in PROJ_KEYS:
        pairs = get_metric(pk, 'svd', metric)
        if not pairs:
            continue
        xs, ys = zip(*pairs)
        ax.plot(xs, ys, marker='o', ms=3, lw=1.5,
                color=ALPHA_COLORS[pk], label=ALPHA_LABELS[pk])
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    if log_y:
        ax.set_yscale('log')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.25)

plt.suptitle('Projection geometry — computed from Ridge W matrix alone (no generation)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Validation metrics: R² and nn_recall@5 per layer, per alpha ───────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (metric, ylabel) in zip(axes, [
    ('projection_r2',   'R² (validation corpus)'),
    ('nn_recall_at_5',  'nn_recall@5 (topology preservation)'),
]):
    for pk in PROJ_KEYS:
        pairs = get_metric(pk, 'validation', metric)
        if not pairs:
            continue
        xs, ys = zip(*pairs)
        ax.plot(xs, ys, marker='o', ms=3, lw=1.5,
                color=ALPHA_COLORS[pk], label=ALPHA_LABELS[pk])
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.25)

plt.suptitle('R² vs nn_recall@5 — these pull in opposite directions; high alpha maximises R² but kills topology',
             fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── R² vs nn_recall scatter: one point per (alpha, layer), coloured by alpha ──
# Shows the trade-off frontier. The "interesting" regime is the upper-right corner
# (high nn_recall AND high R²), which is probably unreachable.
fig, ax = plt.subplots(figsize=(7, 5))

for pk in PROJ_KEYS:
    r2_dict  = dict(get_metric(pk, 'validation', 'projection_r2'))
    nnr_dict = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
    common   = sorted(set(r2_dict) & set(nnr_dict))
    if not common:
        continue
    xs = [r2_dict[l]  for l in common]
    ys = [nnr_dict[l] for l in common]
    sc = ax.scatter(xs, ys, c=[l for l in common], cmap='plasma',
                    s=40, alpha=0.8, label=ALPHA_LABELS[pk],
                    edgecolors=ALPHA_COLORS[pk], linewidths=1.5)

ax.set_xlabel('R² (reconstruction fidelity)')
ax.set_ylabel('nn_recall@5 (topology preservation)')
ax.set_title('R² vs nn_recall@5 trade-off front\nMarker depth = layer (bright = deep layers)')
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# Print the alpha values with best nn_recall at the last layer
last_layer = max(
    int(l) for pk in PROJ_KEYS for l, _ in get_metric(pk, 'validation', 'nn_recall_at_5')
) if any(get_metric(pk, 'validation', 'nn_recall_at_5') for pk in PROJ_KEYS) else None

if last_layer is not None:
    print(f'\nnn_recall@5 at last layer (L{last_layer}) by alpha:')
    for pk in PROJ_KEYS:
        d = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
        v = d.get(last_layer, float('nan'))
        print(f'  {ALPHA_LABELS[pk]:20s}  nn_recall@5 = {v:.4f}')

In [ ]:
# ── Summarise which alphas are present and what they look like ────────────────
print('Alpha summary at last layer:')
print()

last_layer = None
for pk in PROJ_KEYS:
    layers = [l for l, _ in get_metric(pk, 'validation', 'nn_recall_at_5')]
    if layers:
        last_layer = max(last_layer or 0, max(layers))

rows = []
for pk in PROJ_KEYS:
    erank  = dict(get_metric(pk, 'svd', 'erank'))
    nnr    = dict(get_metric(pk, 'validation', 'nn_recall_at_5'))
    layers = sorted(set(erank) & set(nnr))
    if not layers:
        continue
    last = max(layers)
    rows.append((pk, erank.get(min(layers)), erank.get(last), nnr.get(last, float('nan'))))
    print(f'  {ALPHA_LABELS[pk]:20s}  '
          f'erank(L0)={erank.get(min(layers), "?"):>5.0f}  '
          f'erank(L{last})={erank.get(last, "?"):>5.0f}  '
          f'nn_recall(L{last})={nnr.get(last, float("nan")):.3f}')

print()

# Dynamic recommendation: identify lowest-alpha and highest-alpha keys present
if rows:
    best_nnr = max(rows, key=lambda r: r[3])
    worst_nnr = min(rows, key=lambda r: r[3])
    print(f'Best nn_recall@5:   {ALPHA_LABELS[best_nnr[0]]}  (use for gating — most discriminative images)')
    print(f'Worst nn_recall@5:  {ALPHA_LABELS[worst_nnr[0]]}  (corpus prior regime — useful as compression baseline)')
    if best_nnr[0] != worst_nnr[0]:
        print()
        print(f'Recommendation: run gating experiment with at minimum '
              f'{ALPHA_LABELS[best_nnr[0]]} and {ALPHA_LABELS[worst_nnr[0]]}.')
    if len(rows) > 2:
        print(f'Intermediate alpha values ({", ".join(ALPHA_LABELS[r[0]] for r in rows[1:-1])}) '
              f'can be added after gate passes if finer granularity is needed.')

---
## Phase 2 — CLIP-space layer map (zero generation)

Compute two quantities from `manifest["probe_clip_vectors"]` — no SD, no new LLM runs:

| Metric | Formula | What it reveals |
|--------|---------|-----------------|
| **CLIP drift** | `cosine_dist(proj_Ln, proj_L{n-1})` | Layer-to-layer transitions; high drift = boundary between representational regimes |
| **Compression sensitivity** | `cosine_dist(proj_α_low(Ln), proj_α_high(Ln))` | How much regularisation moves each layer's CLIP vector; proxy for corpus peripherality at that layer |

Both feed directly into `INTERESTING_LAYERS` — the set of layers worth running full 16-seed generation on.

**If `probe_clip_vectors` is missing:** run the Phase 3 survey pass first (2 seeds, all layers), then re-run this phase.

In [ ]:
import numpy as np

def cosine_dist(a, b):
    """Cosine distance between two vectors (1 − cosine similarity)."""
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 1.0
    return float(1.0 - np.dot(a, b) / (na * nb))


clip_vecs = manifest.get('probe_clip_vectors', {})

if not clip_vecs:
    print('No probe_clip_vectors in manifest yet.')
    print('Run Phase 3 survey pass first, then re-run Phases 2–4.')
    CLIP_PHASE_READY = False
else:
    CLIP_PHASE_READY = True
    print(f'probe_clip_vectors: {len(clip_vecs)} projection(s)\n')
    for pk, slugs in sorted(clip_vecs.items()):
        layer_counts = [len(v) for v in slugs.values()]
        n_layers = layer_counts[0] if layer_counts else 0
        print(f'  {pk}: {len(slugs)} probes × {n_layers} layers')
        for slug in sorted(slugs)[:6]:
            print(f'    {slug}')
        if len(slugs) > 6:
            print(f'    … and {len(slugs) - 6} more')

In [ ]:
if not CLIP_PHASE_READY:
    print('Skipping — populate probe_clip_vectors first (Phase 3 survey pass).')
    drift_data, comp_sens_data, pk_low, pk_high = {}, {}, None, None
else:
    import re as _re2

    # ── CLIP drift: cosine_dist(L_n, L_{n-1}) for each (proj_key, probe) ──────
    drift_data = {}
    for pk, slugs in clip_vecs.items():
        drift_data[pk] = {}
        for slug, layers in slugs.items():
            layer_indices = sorted(int(k) for k in layers)
            drift = {}
            for i in range(1, len(layer_indices)):
                L_prev, L_curr = layer_indices[i - 1], layer_indices[i]
                drift[L_curr] = cosine_dist(layers[str(L_prev)], layers[str(L_curr)])
            drift_data[pk][slug] = drift

    # ── Compression sensitivity: cosine_dist(α_low, α_high) per (probe, layer) ─
    def _alpha_num(pk):
        m = _re2.search(r'alpha(.+)$', pk)
        if not m: return None
        try: return float(m.group(1))
        except: return None

    numeric_pks = {pk: _alpha_num(pk) for pk in clip_vecs if _alpha_num(pk) is not None}
    comp_sens_data, pk_low, pk_high = {}, None, None

    if len(numeric_pks) >= 2:
        pk_low  = min(numeric_pks, key=numeric_pks.get)
        pk_high = max(numeric_pks, key=numeric_pks.get)
        print(f'Compression sensitivity: α={numeric_pks[pk_low]:.0f}  vs  α={numeric_pks[pk_high]:.0f}')
        common_slugs = set(clip_vecs[pk_low]) & set(clip_vecs[pk_high])
        for slug in sorted(common_slugs):
            vecs_low  = clip_vecs[pk_low][slug]
            vecs_high = clip_vecs[pk_high][slug]
            common_ls = sorted(set(int(k) for k in vecs_low) & set(int(k) for k in vecs_high))
            comp_sens_data[slug] = {
                l: cosine_dist(vecs_low[str(l)], vecs_high[str(l)]) for l in common_ls
            }
        print(f'  {len(comp_sens_data)} probes computed')
    else:
        print('Only one alpha present — compression sensitivity needs ≥2 alpha projections.')

    n_pairs = sum(len(s) for s in drift_data.values())
    print(f'Drift: {n_pairs} (proj × probe) pairs computed')

In [ ]:
if not CLIP_PHASE_READY or not any(drift_data.values()):
    print('Skipping — no drift data.')
else:
    from matplotlib.cm import get_cmap as _gcm

    ALL_SLUGS = sorted(set(s for pk_d in drift_data.values() for s in pk_d))
    _cmap     = _gcm('tab20', max(len(ALL_SLUGS), 1))
    SLUG_COLORS = {s: _cmap(i) for i, s in enumerate(ALL_SLUGS)}

    n_pk  = len(drift_data)
    n_col = 1 + int(bool(comp_sens_data))
    fig, axes = plt.subplots(n_pk, n_col, figsize=(7 * n_col, 3.8 * max(n_pk, 1)), squeeze=False)

    for row, (pk, slug_drifts) in enumerate(sorted(drift_data.items())):
        ax = axes[row, 0]
        for slug, drift in sorted(slug_drifts.items()):
            if not drift: continue
            xs, ys = zip(*sorted(drift.items()))
            ax.plot(xs, ys, marker='o', ms=2.5, lw=1.1,
                    color=SLUG_COLORS.get(slug, '#aaa'), label=slug[:28], alpha=0.85)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Cosine distance L→L−1')
        ax.set_title(f'CLIP drift  [{ALPHA_LABELS.get(pk, pk)}]')
        ax.legend(fontsize=6, ncol=2, loc='upper left')
        ax.grid(True, alpha=0.25)

        if comp_sens_data and n_col > 1:
            ax2 = axes[row, 1]
            for slug, cs in sorted(comp_sens_data.items()):
                if not cs: continue
                xs, ys = zip(*sorted(cs.items()))
                ax2.plot(xs, ys, marker='o', ms=2.5, lw=1.1,
                         color=SLUG_COLORS.get(slug, '#aaa'), label=slug[:28], alpha=0.85)
            lbl_low  = f'α={int(_alpha_num(pk_low))}'  if pk_low  else '?'
            lbl_high = f'α={int(_alpha_num(pk_high))}' if pk_high else '?'
            ax2.set_xlabel('Layer')
            ax2.set_ylabel(f'Cosine dist ({lbl_low} vs {lbl_high})')
            ax2.set_title('Compression sensitivity by layer')
            ax2.legend(fontsize=6, ncol=2, loc='upper left')
            ax2.grid(True, alpha=0.25)

    plt.suptitle('CLIP-space layer map — zero generation', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Score each layer as a combination of:
#   • mean CLIP drift         (high = representational transition)
#   • mean compression sens.  (high = corpus-peripheral activation at that layer)
#   • inverted seed variance  (low var = convergence = interesting for generation)
# Output: INTERESTING_LAYERS for the targeted generation pass.

seed_var_manifest = manifest.get('seed_variance', {})

def _parse_sv_key(key):
    """Parse '{proj_key}/{slug}/L{NNNN}/CFG{v}' → (proj_key, slug, layer, cfg)."""
    parts = key.split('/')
    if len(parts) < 4 or not parts[2].startswith('L'):
        return None
    try:
        return parts[0], parts[1], int(parts[2][1:]), float(parts[3].replace('CFG', ''))
    except ValueError:
        return None

sv_by_layer = {}
for key, rec in seed_var_manifest.items():
    parsed = _parse_sv_key(key)
    if parsed is None: continue
    _, _, layer, _ = parsed
    mpv = rec.get('mean_pixel_var')
    if mpv is not None:
        sv_by_layer.setdefault(layer, []).append(float(mpv))
sv_mean = {l: float(np.mean(vs)) for l, vs in sv_by_layer.items()}

# Aggregate drift across all proj_keys
all_drift_ls = sorted(set(l for pk_d in drift_data.values() for d in pk_d.values() for l in d))
drift_mean = {
    l: float(np.mean([d[l] for pk_d in drift_data.values() for d in pk_d.values() if l in d]))
    for l in all_drift_ls
}
comp_mean = {}
if comp_sens_data:
    all_cs_ls = sorted(set(l for cs in comp_sens_data.values() for l in cs))
    comp_mean = {
        l: float(np.mean([cs[l] for cs in comp_sens_data.values() if l in cs]))
        for l in all_cs_ls
    }

def _norm_dict(d):
    if not d: return {}
    lo, hi = min(d.values()), max(d.values())
    if hi == lo: return {k: 0.5 for k in d}
    return {k: (v - lo) / (hi - lo) for k, v in d.items()}

drift_norm = _norm_dict(drift_mean)
comp_norm  = _norm_dict(comp_mean)
sv_inv     = {l: 1.0 - v for l, v in _norm_dict(sv_mean).items()}  # low var → high score

all_scoreable = sorted(set(drift_norm) | set(comp_norm) | set(sv_inv))
combined_scores = {
    l: float(np.mean([v for v in [drift_norm.get(l), comp_norm.get(l), sv_inv.get(l)]
                      if v is not None]))
    for l in all_scoreable
}

N_TARGET = 8
if all_scoreable:
    L_FIRST, L_LAST = min(all_scoreable), max(all_scoreable)
    top_n = sorted(combined_scores, key=combined_scores.get, reverse=True)[:N_TARGET]
    INTERESTING_LAYERS = sorted(set(top_n) | {L_FIRST, L_LAST})
else:
    INTERESTING_LAYERS = list(range(0, 24, 3))
    print('No metric data — fallback: every 3rd layer.')

print(f'{"Layer":>5}  {"Score":>6}  {"Drift":>7}  {"CompSens":>8}  {"SeedVar↓":>8}  Status')
print('─' * 60)
for l in sorted(combined_scores, key=combined_scores.get, reverse=True)[:16]:
    status = '← targeted' if l in INTERESTING_LAYERS else ''
    print(f'  L{l:2d}   {combined_scores[l]:.3f}  '
          f'{drift_mean.get(l, 0):.4f}  {comp_mean.get(l, 0):.5f}  '
          f'{sv_mean.get(l, 0):8.0f}  {status}')
print(f'\nINTERESTING_LAYERS = {INTERESTING_LAYERS}  ({len(INTERESTING_LAYERS)} layers)')

---
## Phase 3 — Adaptive generation

Two passes, both to `EXISTING_RESULTS` (reuses trained projections):

| Pass | Seeds | Layers | Steps | Purpose |
|------|-------|--------|-------|---------|
| **Survey** | 2 | all | 30 | Populate `probe_clip_vectors` + rough variance trajectory |
| **Targeted** | 16 | `INTERESTING_LAYERS` | 30 | Full measurement at layers that matter |

The manifest is idempotent: survey seeds are not re-generated during the targeted pass.  
Net cost: `(all_layers × 2 + interesting_layers × 14) × n_probes` images.

**Why 30 steps throughout:** between-seed variance is determined by the initial noise sample, not denoising precision.  Fewer steps cause seeds to diverge *more* — imprecise updates miss convergence basins, so the variance measure stops reflecting CLIP-vector specificity and starts reflecting denoiser noise instead.

**On SDXL Turbo:** Turbo uses CFG=0 (no classifier-free guidance), so conditioning differences receive zero amplification.  At CFG=0 the SDXL training distribution prior wins — it maps to indoor/bedroom scenes as the modal image type regardless of probe.  Worth revisiting after the CFG sensitivity floor has been characterised (Exp 4+), but SD 1.5 + CFG=25 is the right baseline here.

In [ ]:
FULL_SEEDS = [42, 123, 777, 456, 888, 321, 654, 987,
              111, 222, 333, 444, 555, 666, 999, 100]

# Pick the proj_key to visualise — lowest alpha (most discriminative)
import re as _re3
def _a_sort(pk):
    m = _re3.search(r'alpha(.+)$', pk)
    if not m: return float('inf')
    try: return float(m.group(1))
    except: return float('inf')

VIZ_PROJ_KEY = min(PROJ_KEYS, key=_a_sort) if PROJ_KEYS else 'per_layer_alpha1'
print(f'VIZ_PROJ_KEY = {VIZ_PROJ_KEY}  (override here if needed)')

targeted_cfg = {
    '_comment': 'Targeted pass — 16 seeds × INTERESTING_LAYERS × 30 steps.',
    'models': base_models,
    'projections': {
        'types': ['per_layer'],
        'alpha_values': [SURVEY_ALPHA],
        'training_data_size': base_cfg.get('projections', {}).get('training_data_size', 5000),
    },
    'probe_texts': SURVEY_PROBES,
    'layers': INTERESTING_LAYERS if INTERESTING_LAYERS else trained_layers,
    'cfg_values': [25.0],
    'seeds': FULL_SEEDS,
    'num_inference_steps': 30,
    'track_lpips': False,
    'output': {'base_dir': str(EXISTING_RESULTS), 'image_format': 'png'},
}
targeted_cfg_path = REPO_DIR / '_targeted_config.json'
targeted_cfg_path.write_text(__import__('json').dumps(targeted_cfg, indent=2))

n_imgs_t = len(targeted_cfg['layers']) * len(FULL_SEEDS) * n_probes
print(f'Targeted config → {targeted_cfg_path}')
print(f'  {len(targeted_cfg["layers"])} layers × {len(FULL_SEEDS)} seeds × {n_probes} probes = ~{n_imgs_t} images')
print(f'  Manifest is idempotent — survey seeds (42, 123) are skipped, only 14 new seeds generated per layer.')
print(f'  Target layers: {targeted_cfg["layers"]}')
print()

RUN_TARGETED = False  # ← set True to execute

if RUN_TARGETED:
    result = subprocess.run(
        ['python', str(REPO_DIR / 'run_experiment.py'),
         '--config', str(targeted_cfg_path), '--phase', 'generate'],
        capture_output=False, text=True, cwd=str(REPO_DIR),
    )
    print('Exit code:', result.returncode)
    with open(EXISTING_RESULTS / 'manifest.json') as f:
        manifest = __import__('json').load(f)
    sv = manifest.get('seed_variance', {})
    print(f'seed_variance entries: {len(sv)}')
    print('Re-run Phase 2 layer-interest-score cell to update INTERESTING_LAYERS with full seed data.')
else:
    print('RUN_TARGETED = False — set True and re-run to execute.')

In [ ]:
import subprocess

base_cfg    = manifest.get('config', {})
base_models = base_cfg.get('models', {
    'llm': 'EleutherAI/pythia-410m',
    'sd':  'sd-legacy/stable-diffusion-v1-5',
    'clip_model': 'ViT-L-14',
    'clip_pretrained': 'openai',
})

# Probe set from existing manifest config, or define here
SURVEY_PROBES = base_cfg.get('probe_texts') or {
    'default': ['a cat', 'a house', 'the color of Tuesday', 'democracy', 'beauty'],
}

# Best-nn_recall alpha from Phase 1 (falls back to 1)
SURVEY_ALPHA = 1
if 'best_nnr' in dir() and rows:
    a = _parse_alpha(best_nnr[0])
    if isinstance(a, (int, float)):
        SURVEY_ALPHA = int(a) if float(a) == int(a) else a

# All layers that were trained
trained_layers = sorted(set(
    int(l) for pk in projections for l in projections[pk].get('svd', {})
)) or list(range(24))

survey_cfg = {
    '_comment': 'Survey pass — 2 seeds × all layers × 30 steps. Populates probe_clip_vectors + rough seed_variance.',
    'models': base_models,
    'projections': {
        'types': ['per_layer'],
        'alpha_values': [SURVEY_ALPHA],
        'training_data_size': base_cfg.get('projections', {}).get('training_data_size', 5000),
    },
    'probe_texts': SURVEY_PROBES,
    'layers': trained_layers,
    'cfg_values': [25.0],
    'seeds': [42, 123],
    'num_inference_steps': 30,
    'track_lpips': False,
    'output': {'base_dir': str(EXISTING_RESULTS), 'image_format': 'png'},
}
survey_cfg_path = REPO_DIR / '_survey_config.json'
survey_cfg_path.write_text(__import__('json').dumps(survey_cfg, indent=2))

n_probes = sum(len(v) for v in SURVEY_PROBES.values())
n_imgs   = len(trained_layers) * len(survey_cfg['seeds']) * n_probes
print(f'Survey config → {survey_cfg_path}')
print(f'  {len(trained_layers)} layers × {len(survey_cfg["seeds"])} seeds × {n_probes} probes = ~{n_imgs} images')
print(f'  Output: {EXISTING_RESULTS}  (reuses trained projections; train is skipped if weights exist)')
print()

RUN_SURVEY = False  # ← set True to execute

if RUN_SURVEY:
    result = subprocess.run(
        ['python', str(REPO_DIR / 'run_experiment.py'),
         '--config', str(survey_cfg_path), '--phase', 'generate'],
        capture_output=False, text=True, cwd=str(REPO_DIR),
    )
    print('Exit code:', result.returncode)
    # Reload manifest to pick up newly populated probe_clip_vectors
    with open(EXISTING_RESULTS / 'manifest.json') as f:
        manifest = __import__('json').load(f)
    clip_vecs = manifest.get('probe_clip_vectors', {})
    print(f'probe_clip_vectors now: {len(clip_vecs)} projection(s)')
    print('Re-run Phases 2–4 cells to update CLIP drift and layer scores.')
else:
    print('RUN_SURVEY = False — set True and re-run to execute.')

---
## Phase 4 — Composite image analysis

For each **(probe, layer)** with ≥ 2 seed images, produce a composite figure:

| Panel | What it shows | When it's useful |
|-------|--------------|-----------------|
| **Seed grid** | All seeds tiled | Full spread of p(image \| CLIP vector) — the primary visual display |
| **Pixel mean** | Average across seeds | Valid at convergence layers (unimodal); blurs into noise at bimodal layers |
| **Variance heatmap** | Per-pixel std → `hot` colourmap | Which spatial regions are stable (conditioning-determined) vs. variable (prior noise) |
| **Attractor means (k=1–3)** | k-means cluster means on 32×32 downsampled pixel vectors | Distinct modes; k=1 = unimodal; k≥2 = multiple visual attractors |

The attractor count is the key quantity that scalar `mean_pixel_var` misses:
- `k=1`, low within-cluster variance → **convergence** (CLIP vector is specific)
- `k=1`, high within-cluster variance → **diffuse prior** (CLIP vector is non-specific)
- `k≥2`, distinct means → **multimodal** (CLIP vector straddles SD attractor boundary)

`VIZ_PROJ_KEY` defaults to the lowest-alpha projection (most discriminative). Change it below if needed.

In [ ]:
# ── Per-(probe, layer) composite figures ──────────────────────────────────────
# For each (slug, layer) in INTERESTING_LAYERS:
#   • seed grid  — all seeds tiled
#   • pixel mean — valid if unimodal; blurs if multimodal (treat as diagnostic)
#   • variance heatmap — spatial map of which pixels are stable vs. random
#   • attractor means — k-means cluster means (k chosen by elbow, max 3)

from PIL import Image as _PIL

COMPOSITE_DIR = EXISTING_RESULTS / 'analysis' / 'composites'
COMPOSITE_DIR.mkdir(parents=True, exist_ok=True)
CFG_VIZ = 25.0

img_base = EXISTING_RESULTS / 'grids' / 'by_projection' / VIZ_PROJ_KEY
if not img_base.exists():
    print(f'No images at {img_base} — run survey + targeted passes first.')
else:
    available_slugs = sorted(d.name for d in img_base.iterdir() if d.is_dir())
    layers_to_viz   = INTERESTING_LAYERS if INTERESTING_LAYERS else list(range(0, 24, 3))

    attractor_count_map = {}  # slug → {layer: k}
    mean_pv_map         = {}  # slug → {layer: mean_pixel_var}

    for slug in available_slugs:
        img_dir = img_base / slug / 'per_layer'
        if not img_dir.exists(): continue
        attractor_count_map[slug] = {}
        mean_pv_map[slug] = {}

        for layer in layers_to_viz:
            paths = sorted(img_dir.glob(f'L{layer:04d}_CFG{CFG_VIZ:g}_seed*.png'))
            if len(paths) < 2: continue

            imgs = np.stack([
                np.array(_PIL.open(p).convert('RGB').resize((256, 256), _PIL.LANCZOS))
                for p in paths
            ])  # (N, 256, 256, 3)

            # Pixel mean + variance heatmap
            mean_img = imgs.mean(axis=0).astype(np.uint8)
            std_map  = imgs.astype(float).std(axis=0).mean(axis=-1)
            std_norm = (std_map - std_map.min()) / (std_map.max() - std_map.min() + 1e-9)
            var_heat = (plt.get_cmap('hot')(std_norm)[:, :, :3] * 255).astype(np.uint8)

            # Attractor clustering (k-means in 32×32 pixel space, k=1..3)
            import warnings
            try:
                from sklearn.cluster import KMeans as _KM
                small = np.stack([
                    np.array(_PIL.fromarray(img).resize((32, 32), _PIL.LANCZOS)).flatten().astype(float) / 255.0
                    for img in imgs
                ])
                inertias = {}
                labels_by_k = {}
                for k in range(1, min(4, len(imgs) + 1)):
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        km = _KM(n_clusters=k, n_init=5, random_state=42).fit(small)
                    inertias[k]    = km.inertia_
                    labels_by_k[k] = km.labels_
                # Elbow: largest relative inertia drop
                best_k = 1
                if len(inertias) > 1:
                    drops = {k: (inertias[k-1] - inertias[k]) / (inertias[1] + 1e-9)
                             for k in range(2, len(inertias) + 1)}
                    if drops and max(drops.values()) > 0.10:
                        best_k = max(drops, key=drops.get)
                labels        = labels_by_k[best_k]
                cluster_means = [imgs[labels == c].mean(axis=0).astype(np.uint8)
                                 for c in range(best_k)]
            except ImportError:
                best_k, labels, cluster_means = 1, np.zeros(len(imgs), int), [mean_img]

            attractor_count_map[slug][layer] = best_k
            sv_key = f'{VIZ_PROJ_KEY}/{slug}/L{layer:04d}/CFG{CFG_VIZ:g}'
            mpv    = manifest.get('seed_variance', {}).get(sv_key, {}).get('mean_pixel_var', np.nan)
            if not np.isnan(mpv): mean_pv_map[slug][layer] = mpv

            # ── Build figure ───────────────────────────────────────────────────
            n_bottom = 2 + best_k
            fig = plt.figure(figsize=(14, 7))
            gs  = fig.add_gridspec(2, max(n_bottom, 3), height_ratios=[1.4, 1])

            # Top row: seed grid (all columns)
            n_cols_grid = min(8, len(imgs))
            n_rows_grid = (len(imgs) + n_cols_grid - 1) // n_cols_grid
            H, W = 256, 256
            grid = _PIL.new('RGB', (W * n_cols_grid, H * n_rows_grid), (30, 30, 30))
            for i, img in enumerate(imgs):
                r, c = divmod(i, n_cols_grid)
                grid.paste(_PIL.fromarray(img), (c * W, r * H))

            ax_grid = fig.add_subplot(gs[0, :])
            ax_grid.imshow(grid)
            ax_grid.set_title(f'{slug}   L{layer}   {len(imgs)} seeds   k={best_k}', fontsize=9)
            ax_grid.axis('off')

            # Bottom: mean | variance | attractor means
            fig.add_subplot(gs[1, 0]).imshow(mean_img); \
                fig.axes[-1].set_title('Pixel mean', fontsize=8); fig.axes[-1].axis('off')
            ax_var = fig.add_subplot(gs[1, 1])
            ax_var.imshow(var_heat)
            mpv_s  = f'{mpv:.0f}' if not np.isnan(mpv) else '?'
            ax_var.set_title(f'Variance map  (mpv={mpv_s})', fontsize=8)
            ax_var.axis('off')
            for ci, cm_img in enumerate(cluster_means):
                n_in = int((labels == ci).sum())
                ax_c = fig.add_subplot(gs[1, 2 + ci])
                ax_c.imshow(cm_img)
                ax_c.set_title(f'Attractor {ci+1}\n({n_in} seeds)', fontsize=8)
                ax_c.axis('off')

            plt.suptitle(f'{VIZ_PROJ_KEY}  |  {slug}  |  L{layer}', fontsize=10)
            plt.tight_layout()
            out_p = COMPOSITE_DIR / f'{slug}_L{layer:03d}_k{best_k}.png'
            fig.savefig(out_p, dpi=100, bbox_inches='tight')
            plt.show()
            plt.close()

    print(f'\nSaved composites → {COMPOSITE_DIR}')
    print(f'attractor_count_map: {sum(len(v) for v in attractor_count_map.values())} (slug, layer) entries')

In [ ]:
# ── Attractor count + seed variance heatmap: probe × layer ────────────────────
# Run after composite-generate has populated attractor_count_map and mean_pv_map.

import matplotlib.colors as _mcolors, warnings

if 'attractor_count_map' not in dir() or not attractor_count_map:
    print('Run composite-generate first to populate attractor_count_map.')
else:
    all_slugs_sorted  = sorted(attractor_count_map)
    all_layers_sorted = sorted(set(l for d in attractor_count_map.values() for l in d))

    A = np.full((len(all_slugs_sorted), len(all_layers_sorted)), np.nan)
    P = np.full_like(A, np.nan)
    li_map = {l: i for i, l in enumerate(all_layers_sorted)}

    for si, slug in enumerate(all_slugs_sorted):
        for l, k in attractor_count_map[slug].items():
            A[si, li_map[l]] = k
        for l, mpv in mean_pv_map.get(slug, {}).items():
            P[si, li_map[l]] = mpv

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(len(all_layers_sorted) * 0.6, 10), 6))

    cmap_k = _mcolors.ListedColormap(['#0f3460', '#e94560', '#f5a623'])
    im1 = ax1.imshow(A, aspect='auto', cmap=cmap_k, vmin=0.5, vmax=3.5, interpolation='nearest')
    ax1.set_xticks(range(len(all_layers_sorted)))
    ax1.set_xticklabels([f'L{l}' for l in all_layers_sorted], fontsize=7, rotation=45)
    ax1.set_yticks(range(len(all_slugs_sorted)))
    ax1.set_yticklabels(all_slugs_sorted, fontsize=7)
    ax1.set_title(f'Attractor count k  [{VIZ_PROJ_KEY}]')
    plt.colorbar(im1, ax=ax1, ticks=[1, 2, 3], label='k attractors')

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        log_P = np.where(P > 0, np.log1p(P), np.nan)
    im2 = ax2.imshow(log_P, aspect='auto', cmap='plasma', interpolation='nearest')
    ax2.set_xticks(range(len(all_layers_sorted)))
    ax2.set_xticklabels([f'L{l}' for l in all_layers_sorted], fontsize=7, rotation=45)
    ax2.set_yticks(range(len(all_slugs_sorted)))
    ax2.set_yticklabels(all_slugs_sorted, fontsize=7)
    ax2.set_title('Mean pixel variance log(1+v)  — lower = convergence')
    plt.colorbar(im2, ax=ax2, label='log(1 + mean_pixel_var)')

    plt.tight_layout()
    out_h = EXISTING_RESULTS / 'analysis' / 'attractor_summary.png'
    out_h.parent.mkdir(exist_ok=True)
    plt.savefig(out_h, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_h}')

---
## Deferred — not yet coded

### Exp 9 — Trajectory fingerprinting (data-driven probe categories)

**Needs:** `seed_variance` for all 24 layers × ≥ 10 probes (targeted pass complete).  
**What:** cluster probes by their 24-dim variance trajectory shape (z-scored; DTW or cosine distance); also cluster by 24-dim compression-sensitivity vector.  
**Post-hoc overlay:** do emergent clusters align with Exp 5 input categories (benchmark/periphery/semantic)?  
**Secondary:** PCA of trajectory matrix — PC1 extremes describe dominant mode of variation.  
Add cells here once targeted pass data exists.

---

### Exp 8 — Corpus-stability test (lowest priority)

**Needs:** all current experiments confirmed stable.  Re-runs `train` on Wikipedia-heavy corpus; re-runs survey pass; compares convergence layer indices.  
**Primary question:** does convergence layer shift with corpus?  Stable = architecture property; shifts = corpus doing representational work.  
Add only after Exps 4–7 complete, to avoid invalidating prior comparisons.